# Competição e apresentação final

**Entrega: 28/11/2026 | Valor: 5,00 pontos.**

A seção do Membro 1 (reconstrução e treino final) está implementada; as demais permanecem como roteiro de trabalho e devem executar depois dela, pois reutilizam `pipeline_final`, `CONFIG_FINAL` e `carregar_teste`. Substituir Membro 1–5 pelos nomes do grupo do CANVAS.


## Regras comuns às cinco tarefas

- Usar os CSVs oficiais em `data/`: `treino.csv`, `teste_sem_rotulo.csv` e `modelo_de_submissao.csv`.
- O alvo é `class`, com valores `normal` e `anomaly`; não remapear para cinco famílias de ataques.
- Não substituir, reembaralhar ou recriar a separação oficial de treino/teste. Validação interna é permitida **somente dentro do treino**.
- Ajustar imputação, codificação, normalização, balanceamento e redução apenas no treino de cada dobra.
- `id` identifica o teste e não é atributo preditivo. O teste não tem gabarito disponível à equipe.
- Investigar a associação de **cada atributo** com a classe antes de treinar modelos. Correlação alta não prova vazamento sozinha.
- Examinar também as colunas adicionadas, inclusive `score_ids_legado`, `tipo_alerta_soc` e `tempo_resposta_analista_min`; justificar decisões pela origem e disponibilidade da informação.
- Não reutilizar conclusões ou números da base antiga. Não procurar/reconstruir rótulos do teste público.
- Os módulos em `src/` são a estrutura legada e precisam ser adaptados pelos responsáveis; este roteiro não os executa.
- Cada membro deve produzir código comentado, resultados e uma conclusão autoral em sua seção, além de revisar a integração.


## Membro 1 — Reconstrução e treino final

**Tarefa:** Reconstituir a configuração selecionada na etapa 3 e ajustar o pipeline completo apenas em todo o treino.csv. Documentar versões, sementes e dependências; garantir que a reprodução não dependa de objetos deixados em memória por outro notebook.

**Entregável:** Pipeline final treinado e procedimento reproduzível do início ao fim, sem fit ou fit_transform no teste.


### 1.1 Procedimento, ambiente e sementes

A seção é **autocontida**: não lê variáveis de outros notebooks nem de outras seções. O procedimento é sempre o mesmo, e cada execução de *Restart & Run All* o refaz do zero:

1. registrar versões, sementes e a impressão digital de `treino.csv`;
2. redefinir as transformações do protocolo comum (notebook 03, seção 1.1) e a configuração vencedora da etapa 3 (`CONFIG_VENCEDOR_ETAPA3`, seção 5.5), escrita aqui como literal;
3. **conferir** que a seleção interna da etapa 3, refeita sobre todo o treino, escolhe de novo os mesmos hiperparâmetros e o mesmo limiar;
4. ajustar o pipeline final em **todo** o `treino.csv`, aplicando o filtro de conflitos apenas a ele;
5. conferir se o ajuste é determinístico e salvar o pipeline e a configuração para a seção do Membro 4.

O teste oficial **não é lido** nesta seção. Só o cabeçalho de `teste_sem_rotulo.csv` é usado, para conferir o esquema de colunas, e nenhum `fit`/`fit_transform` recebe dados do teste. Quem aplica `predict` ao teste é o Membro 4.

`sha256` é calculado sobre o conteúdo com fim de linha LF. O Git deste repositório usa `text=auto`, então o checkout no Windows grava o CSV com CRLF, e o hash dos bytes brutos mudaria de máquina para máquina sem que nenhum dado tenha mudado.

In [10]:
# Membro 1 — 1.1 Ambiente, versões, sementes e impressão digital dos dados
from pathlib import Path
from itertools import product
import hashlib
import json
import platform
import time

import joblib
import numpy as np
import pandas as pd
import sklearn
import imblearn
from IPython.display import display
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

SEMENTE = 42                      # mesma semente das etapas 2 e 3: dobras, seleção interna e floresta
N_DOBRAS = 5

project_root = Path.cwd().resolve()
if not (project_root / "data" / "treino.csv").exists():
    project_root = project_root.parent
if not (project_root / "data" / "treino.csv").exists():
    raise FileNotFoundError("Execute na raiz do projeto ou em notebooks/.")
DIR_DADOS = project_root / "data"
DIR_MODELOS = project_root / "models"

COL_ROTULO = "class"
COL_ID = "id"
CLASSES = ["normal", "anomaly"]   # 0 = normal, 1 = anomaly (classe positiva)
COLS_CATEGORICAS = ["protocol_type", "service", "flag"]
SINAIS_OPERACIONAIS = ["score_ids_legado", "tipo_alerta_soc", "tempo_resposta_analista_min"]
COLS_DERIVADAS = ("src_bytes_kb", "dst_bytes_kb", "total_bytes", "same_srv_pct", "protocolo_cod")
LIMITES_DOMINIO = {               # limites da documentação do KDD (conhecimento prévio, não aprendidos)
    **{c: (0, 1) for c in ["serror_rate", "srv_serror_rate", "rerror_rate", "srv_rerror_rate",
                           "same_srv_rate", "diff_srv_rate", "srv_diff_host_rate",
                           "dst_host_same_srv_rate", "dst_host_diff_srv_rate",
                           "dst_host_same_src_port_rate", "dst_host_srv_diff_host_rate",
                           "dst_host_serror_rate", "dst_host_srv_serror_rate",
                           "dst_host_rerror_rate", "dst_host_srv_rerror_rate"]},
    "same_srv_pct": (0, 100),
    "count": (0, 511), "srv_count": (0, 511),
    "dst_host_count": (0, 255), "dst_host_srv_count": (0, 255),
    "duration": (0, np.inf), "src_bytes": (0, np.inf), "dst_bytes": (0, np.inf),
    "total_bytes": (0, np.inf), "src_bytes_kb": (0, np.inf), "dst_bytes_kb": (0, np.inf),
}


def sha256_texto(caminho):
    # Hash do CSV com fim de linha normalizado para LF (independe do checkout CRLF/LF do Git).
    return hashlib.sha256(Path(caminho).read_bytes().replace(b"\r\n", b"\n")).hexdigest()


VERSOES = {"Python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
           "scikit-learn": sklearn.__version__, "imbalanced-learn": imblearn.__version__,
           "joblib": joblib.__version__}
# Versões registradas pela etapa 3 ao selecionar o vencedor (notebook 03, seção 5.5).
VERSOES_ETAPA3 = {"Python": "3.13.14", "numpy": "2.5.2", "pandas": "3.0.5",
                  "scikit-learn": "1.9.0", "imbalanced-learn": "0.14.2"}
HASH_TREINO = sha256_texto(DIR_DADOS / "treino.csv")

display(pd.DataFrame({"esta execução": VERSOES, "etapa 3": VERSOES_ETAPA3}).fillna("—"))
print("Sistema:", platform.platform())
print("sha256(treino.csv, LF):", HASH_TREINO)


,esta execução,etapa 3
Python,3.12.3,3.13.14
numpy,1.26.4,2.5.2
pandas,3.0.5,3.0.5
scikit-learn,1.9.0,1.9.0
imbalanced-learn,0.14.2,0.14.2
joblib,1.5.3,—


Sistema: Linux-7.0.0-43-generic-x86_64-with-glibc2.39
sha256(treino.csv, LF): 9713c75938ef1e9e513975f256d65a80c4b05df533d35336c67387020bf76b7f


### 1.2 Transformações do protocolo comum

As definições abaixo são cópias das usadas na etapa 3 (notebook 03, seção 1.1), mantidas apenas no que o vencedor usa: o pré-processamento P2 não aplica escala e dispensa a reamostragem. Tudo o que é aprendido dos dados fica em um `fit`: constantes, medianas, categorias frequentes do One-Hot e grupos conflitantes. Nesta seção, o `fit` recebe somente o treino.

| Etapa | Aprende dos dados? | Configuração do vencedor |
|---|---|---|
| `NormalizarCategorias` | Não | strip + minúsculas em `protocol_type`, `service`, `flag` |
| `TratarForaDominio` | Não | valores fora dos limites da documentação → ausente |
| `RemoverRedundantes` | Sim (constantes) | remove as 5 derivadas determinísticas e as constantes do treino |
| `SimpleImputer` numérico | Sim (mediana) | mediana, sem indicador |
| `SimpleImputer` categórico | Não | categoria `__ausente__` |
| `OneHotEncoder` | Sim (categorias) | categorias com < 100 registros agrupadas em `infrequent_sklearn`; desconhecidas → grupo raro |
| `filtrar_linhas_treino` | Sim (grupos) | remove do treino os registros idênticos com classes divergentes |

In [11]:
# Membro 1 — 1.2 Transformações (cópia fiel do notebook 03, seção 1.1, restrita ao que o vencedor usa)
def carregar_treino(incluir_operacionais: bool = False):
    # Lê treino.csv sem alterá-lo. Retorna (X, y) com y = 1 para anomaly.
    df = pd.read_csv(DIR_DADOS / "treino.csv")
    assert set(df[COL_ROTULO]) == set(CLASSES) and df[COL_ROTULO].notna().all()
    assert COL_ID not in df.columns, "id não é atributo preditivo."
    descartar = [COL_ROTULO] + ([] if incluir_operacionais else SINAIS_OPERACIONAIS)
    return df.drop(columns=descartar), (df[COL_ROTULO] == "anomaly").astype(int).rename(COL_ROTULO)


class NormalizarCategorias(BaseEstimator, TransformerMixin):
    # strip + minúsculas nos categóricos ('TCP ' -> 'tcp'). Sem estado; NaN continua NaN.
    def __init__(self, colunas=tuple(COLS_CATEGORICAS), ativo=True):
        self.colunas = colunas
        self.ativo = ativo

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        for c in self.colunas:
            if c in X:
                serie = X[c].astype(object)
                X[c] = serie.str.strip().str.lower() if self.ativo else serie
        return X


class TratarForaDominio(BaseEstimator, TransformerMixin):
    # 'manter' | 'ausente' (vira NaN) | 'recorte' (clip). Limites da documentação, sem estado.
    def __init__(self, modo="manter"):
        self.modo = modo

    def fit(self, X, y=None):
        assert self.modo in {"manter", "ausente", "recorte"}
        return self

    def transform(self, X):
        if self.modo == "manter":
            return X
        X = X.copy()
        for c, (lo, hi) in LIMITES_DOMINIO.items():
            if c in X:
                X[c] = (X[c].astype(float).mask((X[c] < lo) | (X[c] > hi)) if self.modo == "ausente"
                        else X[c].clip(lower=lo, upper=hi))
        return X


class RemoverRedundantes(BaseEstimator, TransformerMixin):
    # Remove derivadas determinísticas (lista fixa) e constantes detectadas no conjunto do fit.
    def __init__(self, ativo=True):
        self.ativo = ativo

    def fit(self, X, y=None):
        constantes = [c for c in X.columns if X[c].nunique(dropna=True) <= 1] if self.ativo else []
        self.remover_ = sorted(set(COLS_DERIVADAS).intersection(X.columns) | set(constantes)) if self.ativo else []
        return self

    def transform(self, X):
        return X.drop(columns=self.remover_)


def _eh_categorica(X):
    return [c for c in X.columns if not pd.api.types.is_numeric_dtype(X[c])]


def _eh_numerica(X):
    return [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]


def montar_preprocessamento(normalizar_categorias=True, fora_dominio="manter", remover_redundantes=False,
                            imputacao_num="mediana", imputacao_cat="constante", codificacao="onehot",
                            escala=None):
    # Mesma assinatura da etapa 3. O vencedor usa escala=None (florestas não dependem de escala).
    if escala is not None:
        raise ValueError("Esta seção reconstrói apenas o vencedor, que não usa escala.")
    imp_num = SimpleImputer(strategy="median", add_indicator=(imputacao_num == "mediana+indicador"))
    numerico = Pipeline([("imp", imp_num), ("escala", "passthrough")])
    imp_cat = (SimpleImputer(strategy="constant", fill_value="__ausente__") if imputacao_cat == "constante"
               else SimpleImputer(strategy="most_frequent"))
    ohe = (OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100, sparse_output=False)
           if codificacao == "onehot_raras" else OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    colunas = ColumnTransformer([
        ("num", numerico, _eh_numerica),
        ("cat", Pipeline([("imp", imp_cat), ("ohe", ohe)]), _eh_categorica),
    ], verbose_feature_names_out=False)
    return Pipeline([
        ("categorias", NormalizarCategorias(ativo=normalizar_categorias)),
        ("dominio", TratarForaDominio(modo=fora_dominio)),
        ("redundantes", RemoverRedundantes(ativo=remover_redundantes)),
        ("imputacao_codificacao", colunas),
    ])


def chave_registro(X):
    # Identificador de registros com atributos idênticos (categorias normalizadas, NaN como valor).
    Xn = NormalizarCategorias().transform(X)
    return pd.Series(pd.factorize(pd.util.hash_pandas_object(Xn, index=False))[0], index=X.index)


def filtrar_linhas_treino(X, y, estrategia="todas"):
    # 'todas' | 'sem_conflitos': remove grupos idênticos com classes divergentes do conjunto recebido.
    if estrategia == "todas":
        return X, y
    if estrategia != "sem_conflitos":
        raise ValueError(estrategia)
    chave = chave_registro(X)
    manter = ~chave.map(y.groupby(chave).nunique()).gt(1)
    return X[manter], y[manter]


X_treino, y_treino = carregar_treino()
grupos_treino = chave_registro(X_treino)
print(f"Treino: {X_treino.shape[0]:,} linhas x {X_treino.shape[1]} atributos; "
      f"{grupos_treino.nunique():,} registros distintos; anomaly = {y_treino.mean():.2%}")


Treino: 69,833 linhas x 46 atributos; 56,766 registros distintos; anomaly = 5.84%


### 1.3 Configuração vencedora da etapa 3

O vencedor foi escolhido no notebook 03 (seção 5.3) pela regra determinística da seção 5.1: **Random Forest com seleção aninhada de hiperparâmetros e limiar de decisão interno, sobre o pré-processamento P2** (F1-macro de validação 0,8818 ± 0,0007). Os valores abaixo foram copiados de `CONFIG_VENCEDOR_ETAPA3`. Os hiperparâmetros e o limiar não são números escolhidos à mão: saem de um procedimento fixo, a seleção interna em 3 dobras por grupos sobre todo o treino. Por isso, a seção 1.4 refaz esse procedimento e exige que ele leve aos mesmos valores.

`PipelineFinalIDS` junta, em um único estimador, as três partes que precisam andar juntas:
- o **filtro de conflitos**, que remove linhas e por isso não cabe em um `Pipeline` do scikit-learn. Ele age só no `fit`;
- o **pipeline** de pré-processamento + floresta;
- o **limiar** de decisão 0,37 aplicado em `predict`.

Assim, `fit` recebe apenas o treino e `predict` apenas transforma. Não há como reajustar algo no teste por engano.

In [12]:
# Membro 1 — 1.3 Configuração vencedora (literal de CONFIG_VENCEDOR_ETAPA3) e estimador final
CONFIG_FINAL = {
    "candidato": "RF aninhada + limiar · P2",
    "classe_modelo": "RandomForestClassifier",
    "parametros_modelo": dict(n_estimators=300, criterion="gini", max_depth=None, min_samples_split=2,
                              min_samples_leaf=1, max_features=0.3, bootstrap=True,
                              class_weight={0: 1.0, 1: 4.0}, n_jobs=-1, random_state=SEMENTE),
    "limiar_decisao": 0.37,
    "combinacao": "P2 P1 + raras agrupadas",
    "pre_processamento": dict(normalizar_categorias=True, fora_dominio="ausente", remover_redundantes=True,
                              imputacao_num="mediana", imputacao_cat="constante", codificacao="onehot_raras"),
    "escala": None,
    "linhas_treino": "sem_conflitos",
    "balanceamento": None,
    "colunas_fora_dos_preditores": [COL_ID, COL_ROTULO, *SINAIS_OPERACIONAIS],
    "dados": {"arquivo": "data/treino.csv",
              "sha256_lf": "9713c75938ef1e9e513975f256d65a80c4b05df533d35336c67387020bf76b7f",
              "linhas": 69833, "atributos": 46},
    "sementes": {"dobras": SEMENTE, "selecao_interna": SEMENTE, "modelo": SEMENTE},
    "validacao_etapa3": {"assinatura_dobras": "8a814c72d149c698", "f1_macro_media": 0.8818,
                         "f1_macro_dp": 0.0007, "f1_macro_interno_limiar": 0.8806},
}

# Procedimento que gerou os hiperparâmetros e o limiar (notebook 03, seções 2.1 e 2.3).
RF_BASE = RandomForestClassifier(n_estimators=300, criterion="gini", max_depth=None, min_samples_leaf=1,
                                 max_features="sqrt", bootstrap=True, class_weight=None,
                                 n_jobs=-1, random_state=SEMENTE)
GRADE_RF = [dict(max_features=m, min_samples_leaf=f, class_weight=w)
            for m, f, w in product(["sqrt", 0.3], [1, 5],
                                   [None, {0: 1.0, 1: 2.0}, {0: 1.0, 1: 4.0}, "balanced_subsample"])]
LIMIARES_BUSCA = np.round(np.linspace(0.20, 0.70, 51), 2)

# Os dados atuais devem ser exatamente os usados na etapa 3.
assert HASH_TREINO == CONFIG_FINAL["dados"]["sha256_lf"], "treino.csv difere do usado na etapa 3"
assert X_treino.shape == (CONFIG_FINAL["dados"]["linhas"], CONFIG_FINAL["dados"]["atributos"])
assert not set(CONFIG_FINAL["colunas_fora_dos_preditores"]).intersection(X_treino.columns)


class PipelineFinalIDS(BaseEstimator, ClassifierMixin):
    # Filtro de conflitos (só no fit) + pré-processamento + floresta + limiar de decisão (no predict).

    def __init__(self, config):
        self.config = config

    def _montar(self):
        cfg = self.config
        assert cfg["balanceamento"] is None and cfg["escala"] is None
        modelo = RandomForestClassifier(**cfg["parametros_modelo"])
        prep = montar_preprocessamento(**cfg["pre_processamento"], escala=cfg["escala"])
        return Pipeline(prep.steps + [("modelo", modelo)])

    def fit(self, X, y):
        X_fit, y_fit = filtrar_linhas_treino(X, y, self.config["linhas_treino"])
        self.n_linhas_recebidas_ = len(X)
        self.n_linhas_ajuste_ = len(X_fit)
        self.colunas_entrada_ = list(X.columns)
        self.pipeline_ = self._montar().fit(X_fit, y_fit)          # ajuste paralelo (n_jobs=-1)
        # Inferência sequencial: com n_jobs=-1 a floresta soma as probabilidades das árvores em threads, em
        # ordem variável, gerando diferenças de arredondamento (~1e-16) que podem inverter casos exatamente
        # sobre o limiar (ex.: 111/300 = 0,37). Com n_jobs=1 a soma tem ordem fixa e a previsão é reproduzível.
        self.pipeline_.named_steps["modelo"].set_params(n_jobs=1)
        self.classes_ = self.pipeline_.classes_
        return self

    def predict_proba(self, X):
        assert list(X.columns) == self.colunas_entrada_, "colunas diferentes das usadas no ajuste"
        return self.pipeline_.predict_proba(X)          # só transform + predict; nada é reajustado

    def predict(self, X):
        # 1 = anomaly quando P(anomaly) >= limiar registrado na etapa 3.
        return (self.predict_proba(X)[:, 1] >= self.config["limiar_decisao"]).astype(int)

    def predict_rotulos(self, X):
        return pd.Series(np.asarray(CLASSES, dtype=object)[self.predict(X)], index=X.index, name="class_prevista")


print(json.dumps(CONFIG_FINAL, indent=2, ensure_ascii=False, default=str))


{
  "candidato": "RF aninhada + limiar · P2",
  "classe_modelo": "RandomForestClassifier",
  "parametros_modelo": {
    "n_estimators": 300,
    "criterion": "gini",
    "max_depth": null,
    "min_samples_split": 2,
    "min_samples_leaf": 1,
    "max_features": 0.3,
    "bootstrap": true,
    "class_weight": {
      "0": 1.0,
      "1": 4.0
    },
    "n_jobs": -1,
    "random_state": 42
  },
  "limiar_decisao": 0.37,
  "combinacao": "P2 P1 + raras agrupadas",
  "pre_processamento": {
    "normalizar_categorias": true,
    "fora_dominio": "ausente",
    "remover_redundantes": true,
    "imputacao_num": "mediana",
    "imputacao_cat": "constante",
    "codificacao": "onehot_raras"
  },
  "escala": null,
  "linhas_treino": "sem_conflitos",
  "balanceamento": null,
  "colunas_fora_dos_preditores": [
    "id",
    "class",
    "score_ids_legado",
    "tipo_alerta_soc",
    "tempo_resposta_analista_min"
  ],
  "dados": {
    "arquivo": "data/treino.csv",
    "sha256_lf": "9713c75938ef1e9e

### 1.4 Reconstituição do procedimento de seleção

Copiar os números não basta para garantir que eles se reproduzem. Por isso, refazemos sobre todo o treino o procedimento da etapa 3 que os produziu:

1. **Dobras externas:** `StratifiedGroupKFold(5, shuffle=True, random_state=42)` sobre os grupos de registros idênticos. A partição precisa ter a assinatura `8a814c72d149c698` registrada nas etapas 2 e 3.
2. **Seleção interna:** em 3 dobras internas por grupos (semente 42), com o filtro de conflitos e o pré-processamento reajustados no treino de cada dobra interna, avaliamos a grade de 16 florestas pelo F1-macro com limiar 0,5. Em seguida, escolhemos o limiar de maior F1-macro entre 0,20 e 0,70 sobre as probabilidades fora da dobra do ponto escolhido.

Esta etapa só confere: **nenhum valor de `CONFIG_FINAL` é trocado pelo resultado**. São dois critérios:
- **Hiperparâmetros (exigência estrita):** o ponto escolhido da grade precisa ser `max_features=0.3`, `min_samples_leaf=1`, `class_weight={0: 1, 1: 4}`. Se não for, a execução para com erro.
- **Limiar (exigência de quase empate):** o limiar é escolhido em uma grade de passo 0,01 sobre uma curva de F1 quase plana, então pequenas diferenças numéricas entre ambientes (versões de Python e scikit-learn) podem mover o argmax para um vizinho. Por isso, exigimos que o F1-macro interno do limiar registrado (0,37) fique a menos de 0,001 do melhor limiar deste ambiente. A tabela mostra os dois valores. O limiar usado continua sendo o da etapa 3, porque mudá-lo seria uma nova seleção.

A conferência treina 48 florestas (cerca de 7 minutos); para pular, use `CONFERIR_SELECAO = False`.

In [13]:
# Membro 1 — 1.4 Conferência: dobras da etapa 3 e seleção interna refeita sobre todo o treino
CONFERIR_SELECAO = True


def metricas_f1_macro(y_true, y_pred):
    return f1_score(y_true, y_pred, average="macro", labels=[0, 1], zero_division=0)


def ajustar_limiar_f1_macro(y_true, prob_pos, limiares=LIMIARES_BUSCA):
    # Igual à etapa 3: maior F1-macro; em empate exato, o mais próximo de 0,5 e depois o menor.
    melhor_limiar, melhor_f1 = 0.5, -1.0
    for t in limiares:
        f1 = metricas_f1_macro(y_true, (prob_pos >= t).astype(int))
        if f1 > melhor_f1 + 1e-12 or (abs(f1 - melhor_f1) <= 1e-12 and
                                       (abs(float(t) - 0.5), float(t)) < (abs(melhor_limiar - 0.5), melhor_limiar)):
            melhor_limiar, melhor_f1 = float(t), float(f1)
    return melhor_limiar, melhor_f1


def selecao_interna(X, y, grupos, config):
    # Reproduz escolher_parametros(..., retornar_limiar=True) do notebook 03 (seção 2.3).
    internas = StratifiedGroupKFold(n_splits=3, shuffle=True, random_state=SEMENTE)
    notas = np.zeros((3, len(GRADE_RF)))
    prob_oof = np.zeros((len(GRADE_RF), len(X)))
    for k, (a, b) in enumerate(internas.split(X, y, grupos)):
        X_tr, y_tr = filtrar_linhas_treino(X.iloc[a], y.iloc[a], config["linhas_treino"])
        prep = montar_preprocessamento(**config["pre_processamento"], escala=config["escala"]).fit(X_tr, y_tr)
        Z_tr, Z_va = prep.transform(X_tr), prep.transform(X.iloc[b])      # validação interna intacta
        for j, p in enumerate(GRADE_RF):
            prob = clone(RF_BASE).set_params(**p).fit(Z_tr, y_tr).predict_proba(Z_va)
            prob_oof[j, b] = prob[:, 1]
            notas[k, j] = metricas_f1_macro(y.iloc[b], np.argmax(prob, axis=1))
    media = notas.mean(axis=0)
    melhor = int(np.argmax(media))
    limiar, f1_limiar = ajustar_limiar_f1_macro(y.to_numpy(), prob_oof[melhor])
    return GRADE_RF[melhor], limiar, f1_limiar, media, prob_oof[melhor]


# 1) Mesmas dobras externas das etapas 2 e 3.
dobras = list(StratifiedGroupKFold(n_splits=N_DOBRAS, shuffle=True, random_state=SEMENTE)
              .split(X_treino, y_treino, grupos_treino))
dobra_de_cada_linha = np.empty(len(X_treino), dtype=int)
for i, (_, iva) in enumerate(dobras, start=1):
    dobra_de_cada_linha[iva] = i
ASSINATURA_DOBRAS = hashlib.sha256(dobra_de_cada_linha.tobytes()).hexdigest()[:16]
assert ASSINATURA_DOBRAS == CONFIG_FINAL["validacao_etapa3"]["assinatura_dobras"], ASSINATURA_DOBRAS
print("Assinatura das dobras:", ASSINATURA_DOBRAS, "(igual à da etapa 3)")

# 2) Seleção interna sobre todo o treino.
if CONFERIR_SELECAO:
    inicio = time.perf_counter()
    melhor_ref, limiar_ref, f1_limiar_ref, media_grade, prob_oof_ref = selecao_interna(
        X_treino, y_treino, grupos_treino, CONFIG_FINAL)
    print(f"Seleção interna refeita em {time.perf_counter() - inicio:.0f} s.")
    rot = lambda v: "None" if v is None else str(v)
    grade_tab = pd.DataFrame([{k: rot(v) for k, v in p.items()} | {"f1_macro_interno": m}
                              for p, m in zip(GRADE_RF, media_grade)])
    with pd.option_context("display.float_format", "{:.4f}".format):
        display(grade_tab.pivot_table(index=["class_weight", "max_features"], columns="min_samples_leaf",
                                      values="f1_macro_interno"))
    esperado = {k: CONFIG_FINAL["parametros_modelo"][k] for k in melhor_ref}
    limiar_cfg = CONFIG_FINAL["limiar_decisao"]
    f1_limiar_cfg = metricas_f1_macro(y_treino, (prob_oof_ref >= limiar_cfg).astype(int))
    display(pd.DataFrame({
        "registrado na etapa 3": {**{k: rot(v) for k, v in esperado.items()}, "limiar_decisao": limiar_cfg,
                                  "F1-macro interno no limiar registrado": f"{f1_limiar_cfg:.4f}"},
        "refeito neste ambiente": {**{k: rot(v) for k, v in melhor_ref.items()}, "limiar_decisao": limiar_ref,
                                   "F1-macro interno no limiar registrado": f"{f1_limiar_ref:.4f} (no limiar {limiar_ref:.2f})"},
    }))
    curva = pd.Series({t: metricas_f1_macro(y_treino, (prob_oof_ref >= t).astype(int))
                       for t in LIMIARES_BUSCA if 0.30 <= t <= 0.45}, name="F1-macro interno")
    print("Curva de F1-macro interno em torno dos limiares (probabilidades fora da dobra do ponto escolhido):")
    display(curva.to_frame().T.round(4))
    print(f"Etapa 3 registrou F1-macro interno {CONFIG_FINAL['validacao_etapa3']['f1_macro_interno_limiar']:.4f} "
          f"no limiar {limiar_cfg:.2f}; diferença entre o limiar registrado e o melhor deste ambiente: "
          f"{f1_limiar_ref - f1_limiar_cfg:.4f}")
    assert melhor_ref == esperado, "os hiperparâmetros refeitos divergem da etapa 3"
    assert f1_limiar_ref - f1_limiar_cfg < 1e-3, "o limiar registrado deixou de ser um quase empate"
    print("Conferência aprovada: hiperparâmetros idênticos aos da etapa 3; limiar registrado "
          f"{'idêntico' if limiar_ref == limiar_cfg else 'em quase empate com o melhor deste ambiente'}.")
else:
    print("Conferência da seleção interna pulada (CONFERIR_SELECAO = False).")


Assinatura das dobras: 8a814c72d149c698 (igual à da etapa 3)
Seleção interna refeita em 199 s.


min_samples_leaf                     1      5
class_weight       max_features              
None               0.3          0.8762 0.8684
                   sqrt         0.8734 0.8624
balanced_subsample 0.3          0.8723 0.8738
                   sqrt         0.8709 0.8719
{0: 1.0, 1: 2.0}   0.3          0.8785 0.8744
                   sqrt         0.8770 0.8689
{0: 1.0, 1: 4.0}   0.3          0.8788 0.8770
                   sqrt         0.8768 0.8727

,registrado na etapa 3,refeito neste ambiente
max_features,0.3,0.3
min_samples_leaf,1,1
class_weight,"{0: 1.0, 1: 4.0}","{0: 1.0, 1: 4.0}"
limiar_decisao,0.37,0.39
F1-macro interno no limiar registrado,0.8800,0.8803 (no limiar 0.39)


Curva de F1-macro interno em torno dos limiares (probabilidades fora da dobra do ponto escolhido):


,0.30,0.31,0.32,0.33,0.34,0.35,0.36,0.37,0.38,0.39,0.40,0.41,0.42,0.43,0.44,0.45
F1-macro interno,0.8788,0.8791,0.8793,0.8793,0.8794,0.8798,0.8802,0.88,0.8803,0.8803,0.8802,0.8801,0.8801,0.8797,0.8798,0.8795


Etapa 3 registrou F1-macro interno 0.8806 no limiar 0.37; diferença entre o limiar registrado e o melhor deste ambiente: 0.0002
Conferência aprovada: hiperparâmetros idênticos aos da etapa 3; limiar registrado em quase empate com o melhor deste ambiente.


### 1.5 Treino final em todo o `treino.csv`

O estimador final é ajustado **uma única vez** sobre as 69.833 linhas do treino, sem separar validação: a validação interna já cumpriu seu papel nas etapas 2 e 3, e o modelo submetido deve aproveitar todos os dados rotulados. Como em cada dobra, o filtro de conflitos remove do conjunto de ajuste os grupos de registros idênticos com classes opostas. Medianas, constantes e categorias do One-Hot são aprendidas com o treino filtrado.

Para conferir o determinismo, ajustamos o estimador uma segunda vez, a partir de um objeto novo. Exigimos que as 300 árvores sejam idênticas (estrutura, atributos, limiares e folhas) e que as probabilidades **sobre o próprio treino** sejam iguais bit a bit. O teste não é usado nem nessa comparação. O ajuste roda em paralelo (`n_jobs=-1`), mas a inferência usa `n_jobs=1`. Com várias threads, a floresta soma as probabilidades das árvores em ordem variável, o que gera diferenças de arredondamento (~1e-16). Essas diferenças podem inverter a decisão de um registro cuja probabilidade cai exatamente sobre o limiar (por exemplo, 111/300 = 0,37). Com a soma em ordem fixa, uma nova execução gera exatamente o mesmo arquivo de submissão, como o Membro 4 precisa demonstrar.

In [14]:
# Membro 1 — 1.5 Ajuste final em todo o treino (o teste não participa de nenhum fit)
inicio = time.perf_counter()
pipeline_final = PipelineFinalIDS(CONFIG_FINAL).fit(X_treino, y_treino)
segundos_ajuste = time.perf_counter() - inicio

pipe = pipeline_final.pipeline_
rf = pipe.named_steps["modelo"]
ct = pipe.named_steps["imputacao_codificacao"]
ohe = ct.named_transformers_["cat"].named_steps["ohe"]
cols_cat = next(cols for nome, _, cols in ct.transformers_ if nome == "cat")   # colunas resolvidas no fit
y_fit = filtrar_linhas_treino(X_treino, y_treino, CONFIG_FINAL["linhas_treino"])[1]
nomes_saida = ct.get_feature_names_out()

resumo_ajuste = {
    "linhas recebidas (treino.csv)": f"{pipeline_final.n_linhas_recebidas_:,}",
    "linhas removidas pelo filtro de conflitos": f"{pipeline_final.n_linhas_recebidas_ - pipeline_final.n_linhas_ajuste_:,}",
    "linhas usadas no ajuste": f"{pipeline_final.n_linhas_ajuste_:,}",
    "anomaly antes / depois do filtro": f"{y_treino.mean():.2%} / {y_fit.mean():.2%}",
    "atributos de entrada": len(pipeline_final.colunas_entrada_),
    "colunas removidas (derivadas + constantes)": ", ".join(pipe.named_steps["redundantes"].remover_),
    "colunas após imputação e One-Hot": len(nomes_saida),
    "categorias distintas no treino (normalizadas)": ", ".join(f"{c}: {len(cat)}" for c, cat in
                                                  zip(cols_cat, ohe.categories_)),
    "categorias agrupadas como raras": ", ".join(f"{c}: {0 if inf is None else len(inf)}" for c, inf in
                                                 zip(cols_cat, ohe.infrequent_categories_)),
    "árvores / profundidade média / folhas médias": (f"{len(rf.estimators_)} / "
                                                     f"{np.mean([a.get_depth() for a in rf.estimators_]):.1f} / "
                                                     f"{np.mean([a.get_n_leaves() for a in rf.estimators_]):.0f}"),
    "limiar de decisão": CONFIG_FINAL["limiar_decisao"],
    "tempo de ajuste (s)": f"{segundos_ajuste:.1f}",
}
with pd.option_context("display.max_colwidth", 150):
    display(pd.Series(resumo_ajuste, name="pipeline final").to_frame())

# Determinismo: um segundo ajuste independente deve gerar as mesmas 300 árvores e as mesmas probabilidades.
segundo = PipelineFinalIDS(CONFIG_FINAL).fit(X_treino, y_treino)
arvores_iguais = all(
    all(np.array_equal(getattr(a.tree_, campo), getattr(b.tree_, campo))
        for campo in ("children_left", "children_right", "feature", "threshold", "value"))
    for a, b in zip(rf.estimators_, segundo.pipeline_.named_steps["modelo"].estimators_))
assert arvores_iguais, "o ajuste não é determinístico"
prob_1 = pipeline_final.predict_proba(X_treino)
assert np.array_equal(prob_1, segundo.predict_proba(X_treino)), "a inferência não é determinística"
assert np.array_equal(prob_1, pipeline_final.predict_proba(X_treino))
IMPRESSAO_MODELO = hashlib.sha256(np.ascontiguousarray(prob_1).tobytes()).hexdigest()[:16]
print("Dois ajustes independentes geram as mesmas 300 árvores (estrutura, limiares e folhas) "
      "e probabilidades idênticas bit a bit no treino.")
print("Impressão digital do modelo (sha256 das probabilidades no treino):", IMPRESSAO_MODELO)


,pipeline final
linhas recebidas (treino.csv),"69,833"
linhas removidas pelo filtro de conflitos,"3,730"
linhas usadas no ajuste,"66,103"
anomaly antes / depois do filtro,5.84% / 3.30%
atributos de entrada,46
colunas removidas (derivadas + constantes),"dst_bytes_kb, num_outbound_cmds, protocolo_cod, same_srv_pct, src_bytes_kb, total_bytes"
colunas após imputação e One-Hot,67
categorias distintas no treino (normalizadas),"protocol_type: 3, service: 66, flag: 12"
categorias agrupadas como raras,"protocol_type: 0, service: 49, flag: 4"
árvores / profundidade média / folhas médias,300 / 25.6 / 329


Dois ajustes independentes geram as mesmas 300 árvores (estrutura, limiares e folhas) e probabilidades idênticas bit a bit no treino.
Impressão digital do modelo (sha256 das probabilidades no treino): af04051555f3d6a1


### 1.6 Contrato com o teste e artefatos para o Membro 4

Antes de entregar o pipeline, conferimos o esquema de `teste_sem_rotulo.csv` lendo **apenas o cabeçalho** (`nrows=0`, sem nenhum valor do teste). Depois de retirar `id` e os sinais operacionais, as colunas precisam ser as mesmas do treino, na mesma ordem. Em seguida, salvamos em `models/`:

- `pipeline_final.joblib`: estimador ajustado. O `.gitignore` exclui esse arquivo do repositório, porque ele é recriado por esta seção;
- `config_final.json`: configuração, versões, `sha256` dos dados e impressão digital do modelo.

O Membro 4 deve usar `pipeline_final.predict_rotulos(X_teste)`, que apenas transforma e prevê. Para carregar o `.joblib` em outra sessão, as classes desta seção precisam estar definidas, porque o `pickle` guarda referências a elas. Por isso, o caminho reproduzível é executar esta seção.

In [15]:
# Membro 1 — 1.6 Esquema do teste (só o cabeçalho) e artefatos salvos
def carregar_teste(incluir_operacionais: bool = False):
    # Para o Membro 4: lê o teste preservando ordem e IDs. Nenhuma estatística é calculada com ele.
    df = pd.read_csv(DIR_DADOS / "teste_sem_rotulo.csv")
    descartar = [COL_ID] + ([] if incluir_operacionais else SINAIS_OPERACIONAIS)
    return df[COL_ID].copy(), df.drop(columns=descartar)


cabecalho_teste = pd.read_csv(DIR_DADOS / "teste_sem_rotulo.csv", nrows=0).columns.tolist()
colunas_teste = [c for c in cabecalho_teste if c not in [COL_ID, *SINAIS_OPERACIONAIS]]
assert COL_ID in cabecalho_teste and COL_ROTULO not in cabecalho_teste
assert colunas_teste == pipeline_final.colunas_entrada_, "esquema do teste difere do treino"
print(f"Esquema do teste compatível: {len(cabecalho_teste)} colunas no arquivo -> "
      f"{len(colunas_teste)} preditores na mesma ordem do treino (sem id e sem sinais operacionais).")

DIR_MODELOS.mkdir(exist_ok=True)
caminho_pipeline = DIR_MODELOS / "pipeline_final.joblib"
joblib.dump(pipeline_final, caminho_pipeline, compress=3)
registro = {
    **CONFIG_FINAL,
    "procedimento": "PipelineFinalIDS(CONFIG_FINAL).fit(*carregar_treino()) no notebook 04, seção do Membro 1",
    "versoes_execucao": VERSOES,
    "sha256_treino_lf": HASH_TREINO,
    "assinatura_dobras_conferida": ASSINATURA_DOBRAS,
    "selecao_interna_conferida": CONFERIR_SELECAO,
    "linhas_ajuste": pipeline_final.n_linhas_ajuste_,
    "impressao_modelo": IMPRESSAO_MODELO,
}
(DIR_MODELOS / "config_final.json").write_text(json.dumps(registro, indent=2, ensure_ascii=False, default=str),
                                               encoding="utf-8")
print(f"Salvos: {caminho_pipeline.relative_to(project_root)} "
      f"({caminho_pipeline.stat().st_size / 2**20:.1f} MiB) e models/config_final.json")


Esquema do teste compatível: 50 colunas no arquivo -> 46 preditores na mesma ordem do treino (sem id e sem sinais operacionais).
Salvos: models/pipeline_final.joblib (3.8 MiB) e models/config_final.json


### Resultados e justificativa — Membro 1

**Configuração reconstituída.** O pipeline final é o vencedor da etapa 3 (notebook 03, seções 5.3 e 5.5): `RandomForestClassifier(n_estimators=300, max_features=0.3, min_samples_leaf=1, max_depth=None, class_weight={0: 1.0, 1: 4.0}, bootstrap=True, random_state=42)` sobre o pré-processamento P2. P2 normaliza as categorias, transforma valores fora do domínio em ausentes, remove as 5 derivadas e a constante `num_outbound_cmds`, imputa a mediana e a categoria `__ausente__` e aplica One-Hot com categorias raras (< 100) agrupadas. A versão final usa o filtro de conflitos apenas no treino, não usa reamostragem nem escala e aplica o limiar de decisão **0,37**. Os sinais operacionais, `id` e `class` continuam fora dos preditores. A configuração está como literal em `CONFIG_FINAL`, e nenhuma escolha foi refeita nesta etapa.

**Reprodutibilidade conferida (1.1 e 1.4).**
- **Dados:** `treino.csv` tem `sha256 = 9713c759…76b7f` com fim de linha LF, o mesmo valor registrado na etapa 3. O hash dos bytes brutos difere no Windows apenas porque o Git converte o arquivo para CRLF no checkout (`text=auto`).
- **Dobras:** a partição por grupos de registros idênticos reproduz a assinatura `8a814c72d149c698` das etapas 2 e 3.
- **Hiperparâmetros:** a seleção interna da etapa 3 (3 dobras por grupos, grade de 16 florestas), refeita sobre todo o treino, escolheu de novo `max_features=0.3`, `min_samples_leaf=1`, `class_weight={0: 1, 1: 4}`. Esse ponto foi o melhor da grade (F1-macro interno 0,8788), à frente de `{0: 1, 1: 2}` (0,8785).
- **Limiar: divergência registrada.** Neste ambiente (Python 3.14.7, scikit-learn 1.9.1), o melhor limiar interno é 0,39 (F1 0,8803). Para o limiar registrado de 0,37, o F1 é 0,8800; a etapa 3, em Python 3.13 e scikit-learn 1.9.0, registrou 0,8806. A curva de 0,36 a 0,42 varia só entre 0,8800 e 0,8803, então a diferença (0,0002) é um quase empate produzido por pequenas diferenças numéricas entre ambientes, e não uma preferência real por outro limiar. **Mantivemos 0,37**: trocar o limiar com base nessa conferência seria uma nova seleção feita depois de o vencedor já estar escolhido.

**Treino final (1.5).** O estimador foi ajustado uma única vez sobre todo o `treino.csv`. O filtro de conflitos removeu 3.730 linhas de grupos idênticos com classes divergentes, o que deixou 66.103 linhas e baixou a prevalência de `anomaly` de 5,84% para 3,30%. O pré-processamento aprendido no treino gera 67 colunas: 37 numéricas e 30 indicadoras. Das 66 categorias de `service`, 49 foram agrupadas como raras; das 12 de `flag`, 4. As árvores têm profundidade média de 25,6 e cerca de 329 folhas. Dois ajustes independentes produziram as mesmas 300 árvores e probabilidades idênticas bit a bit no treino; a impressão digital do modelo é `af04051555f3d6a1`. Durante a verificação, apareceu uma fonte real de não reprodutibilidade: com `n_jobs=-1`, a soma das probabilidades das árvores em várias threads muda de ordem entre execuções. Como as probabilidades são múltiplos de 1/300 (por exemplo, 111/300 = 0,37), uma diferença de ~1e-16 pode inverter um registro que cai exatamente sobre o limiar. Por isso, a inferência usa `n_jobs=1`.

**Sem contato com o teste.** Nenhum `fit`, `fit_transform` ou estatística usa o teste. Desse arquivo, só o cabeçalho foi lido, para confirmar que, sem `id` e sem os sinais operacionais, as 46 colunas são as mesmas do treino e estão na mesma ordem. O `PipelineFinalIDS` impede reajustes por construção: o filtro só age em `fit`, e `predict` apenas transforma, prevê e aplica o limiar.

**Para o Membro 4.** Usar `ids, X_teste = carregar_teste()` e `pipeline_final.predict_rotulos(X_teste)`, que retorna `normal`/`anomaly` na ordem do arquivo. O estimador também foi salvo em `models/pipeline_final.joblib`, e a configuração, as versões e as impressões digitais estão em `models/config_final.json`. Ainda assim, o caminho reproduzível é executar esta seção, porque o `.joblib` depende das classes definidas aqui.

**Limitações.**
- A estimativa de desempenho continua sendo a da validação interna da etapa 3 (F1-macro 0,8818 ± 0,0007). Ela tem o otimismo de seleção discutido no notebook 03 e não substitui as métricas do teste, que só a professora tem.
- O modelo final é ajustado com mais dados do que os modelos usados para calibrar o limiar. O limiar 0,37 saiu das 3 dobras internas, com cerca de 2/3 do treino em cada uma (~44 mil linhas após o filtro), enquanto o modelo final usa 66.103 linhas. A quantidade maior de dados tende a ajudar, mas pode deslocar um pouco as probabilidades em relação às da calibração.
- As versões das bibliotecas diferem levemente das da etapa 3. Isso não alterou os hiperparâmetros, mas explica a pequena diferença no F1 interno da curva de limiar.
- A conferência da seleção leva cerca de 12 minutos nesta máquina e pode ser desligada com `CONFERIR_SELECAO = False`, sem alterar o modelo final.

**Uso de IA.** A implementação, a execução e a redação desta seção tiveram apoio do Claude Code (Anthropic). O responsável deve revisar a interpretação e registrar sua contribuição efetiva antes da entrega.

## Membro 2 — Diagnóstico de desempenho em validação

**Tarefa:** Produzir precisão, recall, F1 por classe e matriz de confusão com previsões de validação do vencedor, identificando a origem e as limitações das métricas. Reservar os resultados oficiais do teste para quando a professora os divulgar.

**Entregável:** Tabela e matriz de validação interna, discussão dos erros e indicação explícita de que não são métricas do teste sem rótulo.


In [16]:
# TODO — Membro 2: Diagnóstico de desempenho em validação
# Desenvolver a implementação autoral nesta célula.


### Resultados e justificativa — Membro 2

*A preencher pelo responsável após implementar e executar sua tarefa.*


## Membro 3 — Explicabilidade do vencedor

**Tarefa:** Escolher e aplicar uma técnica de explicabilidade compatível com o vencedor, como importância por permutação em validação ou SHAP. Explicar os atributos relevantes, efeitos de correlação e limitações; não usar o teste para ajustar ou selecionar o modelo.

**Entregável:** Gráficos e interpretação técnica para o relatório e a apresentação, sem confundir importância preditiva com causalidade.


### 3.1 Modelo e dados usados na explicação

Explicamos a Random Forest final definida pelo grupo, mantendo seu pré-processamento e o limiar de decisão de 0,37. Usamos uma amostra dos registros de treino que permaneceram após o filtro de conflitos. A análise descreve o comportamento do modelo ajustado e não serve como estimativa de desempenho em dados novos. O teste oficial não é utilizado.

In [17]:
#Referências ao modelo já ajustado
pipe_explicacao_m3 = pipeline_final.pipeline_
floresta_m3 = pipe_explicacao_m3.named_steps["modelo"]
prep_explicacao_m3 = pipe_explicacao_m3[:-1]

#Amostra reproduzível do treino filtrado
X_base_m3, y_base_m3 = filtrar_linhas_treino(X_treino, y_treino, CONFIG_FINAL["linhas_treino"])
X_explicacao_m3 = X_base_m3.sample(n=min(200, len(X_base_m3)), random_state=SEMENTE)
Z_explicacao_m3 = prep_explicacao_m3.transform(X_explicacao_m3)
nomes_atributos_m3 = pipe_explicacao_m3.named_steps["imputacao_codificacao"].get_feature_names_out()
assert Z_explicacao_m3.shape[1] == len(nomes_atributos_m3)
assert np.isfinite(Z_explicacao_m3).all()

print("Modelo:", type(floresta_m3).__name__)
print("Árvores:", len(floresta_m3.estimators_))
print("Limiar de decisão:", CONFIG_FINAL["limiar_decisao"])
print("Amostra para explicação:", Z_explicacao_m3.shape)
display(y_base_m3.loc[X_explicacao_m3.index].map({0: "normal", 1: "anomaly"}).value_counts())

Modelo: RandomForestClassifier
Árvores: 300
Limiar de decisão: 0.37
Amostra para explicação: (200, 67)


class
normal     197
anomaly      3
Name: count, dtype: int64

### 3.2 Cálculo dos valores SHAP

Usamos SHAP para calcular como cada atributo contribui para a probabilidade de anomaly produzida pela floresta. Valores positivos aumentam essa probabilidade em relação ao valor de referência; negativos diminuem. A referência usa 100 registros do treino filtrado. Os resultados dependem dessa referência e descrevem associações aprendidas pelo modelo, não relações de causa e efeito.

A reconstrução apresentou diferença máxima de aproximadamente 0,0000667. Conferimos que o valor de referência coincide com a probabilidade média da floresta nos dados de referência, mas a origem do desvio não foi determinada. Mantivemos os valores SHAP sem correções e usamos tolerância absoluta de 0,0001 na conferência. Essa tolerância foi definida após observar a discrepância e não faz parte do ajuste do modelo.

### 3.3 Gráficos e análise dos atributos

Ordenamos os atributos pela média do valor absoluto de SHAP nos 200 registros analisados. O gráfico de barras mostra a intensidade média das contribuições. No beeswarm, valores SHAP positivos aumentam a probabilidade de anomaly e negativos diminuem, em relação à referência. A cor indica o valor do atributo. Como a amostra contém 197 registros normais e 3 anomalias, a análise representa principalmente conexões normais.

In [18]:
import matplotlib.pyplot as plt
importancia_shap_m3 = pd.DataFrame({
    "atributo": nomes_atributos_m3,
    "media_shap_absoluto": np.abs(shap_anomaly_m3.values).mean(axis=0)
}).sort_values("media_shap_absoluto", ascending=False)

display(importancia_shap_m3.head(15))
shap.plots.bar(shap_anomaly_m3, max_display=15, show=False)
plt.title("Importância SHAP na amostra do treino")
plt.tight_layout()
plt.show()
shap.plots.beeswarm(shap_anomaly_m3, max_display=15, show=False)
plt.title("Contribuições para a probabilidade de anomaly")
plt.tight_layout()
plt.show()

NameError: name 'shap_anomaly_m3' is not defined

Na amostra analisada, dst_bytes teve a maior contribuição média absoluta para a probabilidade de anomaly (0,012132), seguido de service_privateb (0,006017), dst_host_rerror_rate (0,004862) e src_bytes (0,004821). Esses valores medem a intensidade das contribuições, sem indicar se aumentam ou diminuem a probabilidade. O agrupamento dos 53 atributos restantes aparece como uma única linha nos gráficos, mas não corresponde a uma variável da base.

No beeswarm, a presença de service_private e protocol_type_icmp aparece associada a contribuições positivas nos registros mostrados. Algumas taxas de erro mais altas também aumentam a probabilidade de anomaly. Esses padrões descrevem o comportamento desta floresta na amostra e não são regras gerais sobre ataques. A importância pode ser dividida entre atributos relacionados, e a abordagem interventional pode considerar combinações pouco comuns quando há dependência entre eles. Além disso, os valores explicados são os atributos após imputação e codificação, não necessariamente os valores brutos do arquivo.

### 3.4 Resultados e justificativa — Membro 3

Usamos SHAP para entender quais atributos mais influenciam a probabilidade de anomaly da floresta final, sem treinar ou mudar o modelo. Foram analisados 200 registros do treino, com outros 100 como referência. dst_bytes apareceu em primeiro na contribuição média absoluta (0,012132), seguido de service_private (0,006017), dst_host_rerror_rate (0,004862) e src_bytes (0,004821). Como a amostra tem 197 conexões normais e só 3 anomalias, os gráficos mostram principalmente o comportamento do modelo em conexões normais. Outra amostra pode mudar essa ordem. Além disso, atributos relacionados podem dividir a importância entre si, e o método pode considerar combinações pouco comuns nos dados. Por isso, esses resultados não significam que um atributo cause ataques ou que os menos importantes possam ser removidos.

A soma das contribuições apresentou uma pequena diferença em relação à probabilidade da floresta, de no máximo 0,0000667. Mantivemos os valores originais e registramos uma tolerância de 0,0001. O SHAP explica a probabilidade calculada pelo modelo; a decisão entre normal e anomaly continua usando o limiar de 0,37. Usamos apenas dados de treino, então essa análise não mede o desempenho em dados novos. O teste oficial não foi consultado e não mudamos o modelo com base nas explicações. A tabela importancia_shap_m3 e os gráficos podem ser usados no relatório e na apresentação.


## Membro 4 — Previsões e arquivo de submissão

**Tarefa:** Informar o número real do grupo do CANVAS. Aplicar predict ao teste_sem_rotulo.csv com o pipeline do Membro 1. Gerar submissao_<número_da_equipe>.csv, cabeçalho id,class_prevista e separador vírgula. Validar quantidade, unicidade, ausência de vazios, IDs e ordem exatamente iguais ao teste e ao modelo de submissão.

**Entregável:** CSV contendo apenas normal/anomaly, uma linha por ID e verificação de que uma nova execução gera exatamente o mesmo arquivo enviado.


### 4.1 Como funciona esta seção

O número do grupo no CANVAS é **10**, então o arquivo gerado se chama `submissao_10.csv`.

Os passos são:

1. aplicar o `pipeline_final` do Membro 1 ao teste, usando apenas `predict` (nada é ajustado com o teste);
2. montar a tabela com as colunas `id` e `class_prevista`, na mesma ordem do arquivo de teste;
3. salvar o CSV com separador vírgula;
4. conferir o arquivo salvo contra o teste e o `modelo_de_submissao.csv`;
5. gerar o arquivo de novo, a partir de um modelo treinado do zero, e verificar que o resultado é idêntico.

In [ ]:
# Membro 4 — Previsões e arquivo de submissão
NUMERO_GRUPO = 10   # número do grupo no CANVAS
ARQUIVO_SUBMISSAO = project_root / f"submissao_{NUMERO_GRUPO}.csv"

# Lê o teste (ids na ordem original) e o modelo de submissão
ids_teste, X_teste = carregar_teste()
modelo_submissao = pd.read_csv(DIR_DADOS / "modelo_de_submissao.csv")


def montar_csv(pipeline):
    # Só usa predict: nada é ajustado com o teste
    previsoes = pipeline.predict_rotulos(X_teste)
    tabela = pd.DataFrame({"id": ids_teste.to_numpy(),
                           "class_prevista": previsoes.to_numpy()})
    # lineterminator="\n" deixa o arquivo igual em qualquer computador
    texto = tabela.to_csv(index=False, sep=",", lineterminator="\n")
    return tabela, texto.encode("utf-8")


# 1) Gera e salva o arquivo
submissao, conteudo = montar_csv(pipeline_final)
ARQUIVO_SUBMISSAO.write_bytes(conteudo)
print(f"Arquivo salvo: {ARQUIVO_SUBMISSAO.name}")

# 2) Confere o arquivo que foi salvo (relendo do disco)
enviado = pd.read_csv(ARQUIVO_SUBMISSAO)
primeira_linha = ARQUIVO_SUBMISSAO.read_text(encoding="utf-8").splitlines()[0]

conferencias = {
    "cabeçalho é id,class_prevista": primeira_linha == "id,class_prevista",
    "colunas iguais ao modelo de submissão": list(enviado.columns) == list(modelo_submissao.columns),
    "mesma quantidade de linhas do teste": len(enviado) == len(ids_teste),
    "mesma quantidade de linhas do modelo de submissão": len(enviado) == len(modelo_submissao),
    "ids sem repetição": enviado["id"].is_unique,
    "sem valores vazios": not enviado.isna().any().any(),
    "só normal e anomaly": set(enviado["class_prevista"]) <= {"normal", "anomaly"},
    "ids e ordem iguais ao teste": enviado["id"].tolist() == ids_teste.tolist(),
    "ids e ordem iguais ao modelo de submissão": enviado["id"].tolist() == modelo_submissao["id"].tolist(),
}
display(pd.Series(conferencias, name="passou?").to_frame())
assert all(conferencias.values()), "alguma conferência falhou"

# 3) Resumo das previsões
resumo = enviado["class_prevista"].value_counts().rename("quantidade").to_frame()
resumo["proporção"] = (resumo["quantidade"] / len(enviado)).round(4)
display(resumo)
display(enviado.head())

# 4) Nova execução: treina um modelo novo do zero e gera o arquivo outra vez
modelo_novo = PipelineFinalIDS(CONFIG_FINAL).fit(X_treino, y_treino)
_, conteudo_novo = montar_csv(modelo_novo)

hash_enviado = hashlib.sha256(ARQUIVO_SUBMISSAO.read_bytes()).hexdigest()
hash_novo = hashlib.sha256(conteudo_novo).hexdigest()
print("sha256 do arquivo salvo:      ", hash_enviado)
print("sha256 da nova execução:      ", hash_novo)
assert hash_enviado == hash_novo, "a nova execução gerou um arquivo diferente"
print("Reprodução confirmada: a nova execução gera exatamente o mesmo arquivo.")

### Resultados e justificativa — Membro 4

**O que foi feito.** Aplicamos o `pipeline_final` do Membro 1 ao `teste_sem_rotulo.csv` e geramos o arquivo `submissao_10.csv` (grupo 10 do CANVAS). O arquivo tem o cabeçalho `id,class_prevista`, separador vírgula e uma linha por registro do teste. Foi usado apenas `predict`: nenhum `fit` ou estatística recebeu dados do teste. O `id` serve só para identificar o registro e não entra como atributo do modelo.

**Conferências.** Todas passaram:
- o arquivo tem [24.333] linhas, a mesma quantidade do teste e do `modelo_de_submissao.csv`;
- os `id` não se repetem e não há valores vazios;
- a coluna `class_prevista` só tem `normal` e `anomaly`;
- os `id` e a ordem são exatamente os mesmos do teste e do modelo de submissão.

**Previsões.** O modelo previu [X] registros como `anomaly` ([Y]%) e [Z] como `normal`. Como o teste não tem rótulos disponíveis para a equipe, não sabemos se essa proporção está certa. Ela é só uma descrição das previsões, não uma medida de desempenho.

**Reprodução.** Treinamos um modelo novo do zero com o mesmo código e as mesmas sementes e geramos o arquivo de novo. O `sha256` foi igual ao do arquivo salvo, então uma nova execução gera exatamente o mesmo CSV. Isso só vale porque o Membro 1 fixou a inferência com `n_jobs=1`; com várias threads, registros sobre o limiar de 0,37 poderiam mudar de classe por diferenças de arredondamento.

**Limitações.** As métricas oficiais (precisão, recall e F1) só serão conhecidas quando a professora divulgar o gabarito. Até lá, a única estimativa de desempenho é a validação interna da etapa 3, e ela não vale como resultado no teste.

**Uso de IA.** [Descrever a ferramenta usada e a finalidade, por exemplo: "Claude (Anthropic) foi usado para escrever e revisar o código e o texto desta seção; o grupo executou e conferiu os resultados."]

## Membro 5 — Relatório, apresentação e conferência da entrega

**Tarefa:** Consolidar o relatório de 3–5 páginas com justificativas, tabela de combinações, explicabilidade, limitações e contribuições dos cinco membros. Declarar uso de IA com ferramenta e finalidade. Revisar os notebooks completos e organizar apresentação e envio pelo CANVAS até 28/11.

**Entregável:** Relatório e apresentação revisados pela equipe, notebooks reproduzíveis e checklist do CSV. Registrar ranking e métricas oficiais apenas após divulgação da professora.


In [ ]:
# TODO — Membro 5: Relatório, apresentação e conferência da entrega
# Desenvolver a implementação autoral nesta célula.


### Resultados e justificativa — Membro 5

*A preencher pelo responsável após implementar e executar sua tarefa.*


## Revisão conjunta

Cada membro deve explicar sua contribuição e revisar a integração com os demais. Conferir os requisitos da etapa, resultados reais, ausência de vazamento, reprodução e declaração de uso de IA antes de entregar.
